# Proyecto III - Análisis de datos Olist

## Análisis y limpieza de datos

**Autores:** Javier, Dani y Jesús

**Tecnologías:** Python · Pandas · SQL · MySQL · Jupyter Notebook


In [16]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

In [17]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [18]:
import os
from dotenv import load_dotenv

load_dotenv() #Cargar el .env

True

In [19]:
print("Host:", os.getenv("DB_HOST"))
print("Puerto:", os.getenv("DB_PORT"))
print("Usuario:", os.getenv("DB_USER"))
print("Base de datos:", os.getenv("DB_NAME"))

Host: localhost
Puerto: 3306
Usuario: root
Base de datos: olist


In [20]:
from sqlalchemy import create_engine

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@"
    f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
) #Crear conexión 

In [21]:
with engine.connect() as connection:
    print("Conexión correcta")

Conexión correcta


Notebook → .env → SQLAlchemy → PyMySQL → MySQL ✅

In [22]:
query = "SHOW TABLES;"

tablas = pd.read_sql(query, engine)

tablas #Comprobar tablas 

,Tables_in_olist
0,catalogo_productos
1,categoria_traduccion
2,customers
3,geolocation
4,order_items
5,order_payments
6,order_reviews
7,orders
8,products
9,sellers


In [26]:
query = """
SELECT *
FROM catalogo_productos;
"""

df_catalogo = pd.read_sql(query, engine)

df_catalogo.head()

,product_id,categoria,precio,coste_envio,vendedor_id
0,018ca97302e4293050cc41413194bb19,agro_industry_and_commerce,88.0,11.95,6481e96574816ead57975da2c0f6d80d
1,026f43af35e7951067097527d5c31bcc,agro_industry_and_commerce,179.9,31.34,269cff2d3c8d205c11f37a52402ea93b
2,026f43af35e7951067097527d5c31bcc,agro_industry_and_commerce,179.9,31.34,269cff2d3c8d205c11f37a52402ea93b
3,07f01b6fcacc1b187a71e5074199db2d,agro_industry_and_commerce,57.0,29.99,6481e96574816ead57975da2c0f6d80d
4,0a0adf0de1769b2970f3096d0b5abfb7,agro_industry_and_commerce,58.5,29.11,6481e96574816ead57975da2c0f6d80d


In [27]:
print("Filas:", len(df_catalogo))
print("Columnas:", len(df_catalogo.columns))
print("Product_id únicos:", df_catalogo["product_id"].nunique())

Filas: 112650
Columnas: 5
Product_id únicos: 32951


In [28]:

duplicados = (
    df_catalogo
    .groupby("product_id")
    .size()
    .sort_values(ascending=False)
)

duplicados.head(10)

product_id
aca2eb7d00ea1a7b8ebd4e68314663af    527
99a4788cb24856965c36a24e339b6058    488
422879e10f46682990de24d770e7f83d    484
389d119b48cf3043d311335e499d9c6b    392
368c6c730842d78016ad823897a372db    388
53759a2ecddad2bb87a079a1f1519f73    373
d1c427060a0f73f6b889a5c7c61f2ac4    343
53b36df67ebb7c41585e8d54d6772e08    323
154e7e31ebfa092203795c972e5804a6    281
3dd2a17168ec895c781a9191c1e95ad7    274
dtype: int64

In [29]:
query = "DESCRIBE catalogo_productos;"

estructura_catalogo = pd.read_sql(query, engine)

estructura_catalogo

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(41),NO,,None,
1,categoria,varchar(50),YES,,None,
2,precio,"decimal(10,2)",YES,,None,
3,coste_envio,"decimal(10,2)",YES,,None,
4,vendedor_id,varchar(41),YES,,None,


In [30]:
consulta = """
SELECT
    product_id,
    COUNT(*) AS registros,
    COUNT(DISTINCT vendedor_id) AS vendedores
FROM catalogo_productos
GROUP BY product_id
ORDER BY registros DESC
LIMIT 10;
"""

comprobacion = pd.read_sql(consulta, engine)

comprobacion

,product_id,registros,vendedores
0,aca2eb7d00ea1a7b8ebd4e68314663af,527,1
1,99a4788cb24856965c36a24e339b6058,488,2
2,422879e10f46682990de24d770e7f83d,484,1
3,389d119b48cf3043d311335e499d9c6b,392,1
4,368c6c730842d78016ad823897a372db,388,1
5,53759a2ecddad2bb87a079a1f1519f73,373,1
6,d1c427060a0f73f6b889a5c7c61f2ac4,343,1
7,53b36df67ebb7c41585e8d54d6772e08,323,3
8,154e7e31ebfa092203795c972e5804a6,281,1
9,3dd2a17168ec895c781a9191c1e95ad7,274,1


In [31]:
consulta = """
SELECT *
FROM catalogo_productos
WHERE product_id = 'aca2eb7d00ea1a7b8ebd4e68314663af';
"""

producto = pd.read_sql(consulta, engine)

producto.head(20)

,product_id,categoria,precio,coste_envio,vendedor_id
0,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,69.9,0.00,955fee9216a65b617aa5c0531780ce60
1,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,75.0,20.02,955fee9216a65b617aa5c0531780ce60
2,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,69.9,17.40,955fee9216a65b617aa5c0531780ce60
3,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,75.0,13.08,955fee9216a65b617aa5c0531780ce60
4,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,75.0,13.08,955fee9216a65b617aa5c0531780ce60
5,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,69.9,0.00,955fee9216a65b617aa5c0531780ce60
6,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,69.9,20.06,955fee9216a65b617aa5c0531780ce60
7,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,69.9,24.71,955fee9216a65b617aa5c0531780ce60
8,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,75.0,20.02,955fee9216a65b617aa5c0531780ce60
9,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,69.9,19.98,955fee9216a65b617aa5c0531780ce60


In [32]:
comprobacion = (
    df_catalogo
    .groupby("product_id")
    .agg(
        categorias=("categoria", "nunique"),
        vendedores=("vendedor_id", "nunique")
    )
)

comprobacion.describe()



,categorias,vendedores
count,32951.000000,32951.000000
mean,0.981093,1.045431
std,0.136198,0.260092
min,0.000000,1.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,1.000000,1.000000
max,1.000000,8.000000


In [33]:
comprobacion[comprobacion["vendedores"] > 1].shape

(1225, 2)

In [34]:
consulta = """
SELECT
    product_id,
    COUNT(DISTINCT vendedor_id) AS vendedores
FROM catalogo_productos
GROUP BY product_id
HAVING COUNT(DISTINCT vendedor_id) > 1
ORDER BY vendedores DESC
LIMIT 10;
"""

productos_varios_vendedores = pd.read_sql(consulta, engine)

productos_varios_vendedores

,product_id,vendedores
0,69455f41626a745aea9ee9164cb9eafd,8
1,d285360f29ac7fd97640bf0baef03de0,8
2,36f60d45225e60c7da4558b070ce4b60,7
3,4298b7e67dc399c200662b569563a2b2,7
4,656e0eca68dcecf6a31b8ececfabe3e8,7
5,dbb67791e405873b259e4656bf971246,6
6,461f43be3bdf8844e65b62d9ac2c7a5a,5
7,53ea9da485f6aed8a6f03a85831fe021,5
8,7dd3fec3502f7ef333da2a73e00db394,5
9,be0dbdc3d67d55727a65d4cd696ca73c,5


In [ ]:
ventas_producto = (
    df_catalogo
    .groupby("product_id")
    .size()
    .reset_index(name="unidades_vendidas")
)

ventas_producto.head()
#variable nueva: unidades_vendidas = número de veces que aparece cada product_id en las ventas.

,product_id,unidades_vendidas
0,00066f42aeeb9f3007548bb9d3f33c38,1
1,00088930e925c41fd95ebfe695fd2655,1
2,0009406fd7479715e4bef61dd91f2462,1
3,000b8f95fcb9e0096488278317764d19,2
4,000d9be29b5207b54e86aa1b1ac54872,1


In [36]:
precios_producto = (
    df_catalogo
    .groupby("product_id")
    .agg(
        precio_medio=("precio", "mean"),
        precio_minimo=("precio", "min"),
        precio_maximo=("precio", "max")
    )
    .reset_index()
)

precios_producto.head()
#Precio medio de cada producto


,product_id,precio_medio,precio_minimo,precio_maximo
0,00066f42aeeb9f3007548bb9d3f33c38,101.65,101.65,101.65
1,00088930e925c41fd95ebfe695fd2655,129.90,129.90,129.90
2,0009406fd7479715e4bef61dd91f2462,229.00,229.00,229.00
3,000b8f95fcb9e0096488278317764d19,58.90,58.90,58.90
4,000d9be29b5207b54e86aa1b1ac54872,199.00,199.00,199.00


In [37]:
precio_habitual = (
    df_catalogo
    .groupby(["product_id", "precio"])
    .size()
    .reset_index(name="veces")
    .sort_values(["product_id", "veces"], ascending=[True, False])
)

precio_habitual.head(10)

,product_id,precio,veces
0,00066f42aeeb9f3007548bb9d3f33c38,101.65,1
1,00088930e925c41fd95ebfe695fd2655,129.90,1
2,0009406fd7479715e4bef61dd91f2462,229.00,1
3,000b8f95fcb9e0096488278317764d19,58.90,2
4,000d9be29b5207b54e86aa1b1ac54872,199.00,1
5,0011c512eb256aa0dbbb544d8dffcf6e,52.00,1
6,00126f27c813603687e6ce486d909d01,249.00,2
7,001795ec6f1b187d37335e1c4704762e,38.90,9
8,001b237c0e9bb435f2e54071129237e9,78.90,1
9,001b72dfd63e9833e8c02742adf472e3,34.99,14


In [38]:
# Precio más habitual de cada producto
precio_habitual = (
    df_catalogo
    .groupby(["product_id", "precio"])
    .size()
    .reset_index(name="veces")
    .sort_values(["product_id", "veces"], ascending=[True, False])
    .drop_duplicates("product_id")
)

# Categoría de cada producto
categoria_producto = (
    df_catalogo[["product_id", "categoria"]]
    .drop_duplicates("product_id")
)

# Número de unidades vendidas
ventas_producto = (
    df_catalogo
    .groupby("product_id")
    .size()
    .reset_index(name="unidades_vendidas")
)

# DataFrame final
df_final = (
    precio_habitual[["product_id", "precio"]]
    .merge(categoria_producto, on="product_id", how="left")
    .merge(ventas_producto, on="product_id", how="left")
)

df_final.head()

,product_id,precio,categoria,unidades_vendidas
0,00066f42aeeb9f3007548bb9d3f33c38,101.65,perfumery,1
1,00088930e925c41fd95ebfe695fd2655,129.90,auto,1
2,0009406fd7479715e4bef61dd91f2462,229.00,bed_bath_table,1
3,000b8f95fcb9e0096488278317764d19,58.90,housewares,2
4,000d9be29b5207b54e86aa1b1ac54872,199.00,watches_gifts,1


In [ ]:
print("Filas:", len(df_final))
print("Columnas:", len(df_final.columns))
print("Product_id únicos:", df_final["product_id"].nunique())
print("Valores nulos:")
print(df_final.isnull().sum())

#Comprobación final (productos, columnas. duplicados, nulos)

Filas: 32951
Columnas: 4
Product_id únicos: 32951
Valores nulos:
product_id             0
precio                 0
categoria            623
unidades_vendidas      0
dtype: int64


In [40]:
df_final["categoria"] = df_final["categoria"].fillna("sin_categoria")

In [41]:
total_unidades = df_final["unidades_vendidas"].sum()

print("Total de unidades vendidas:", total_unidades)
#Total de unidades vendidas

Total de unidades vendidas: 112650


In [42]:
import os

os.makedirs("data", exist_ok=True)

df_final.to_csv(
    "dt2_productos.csv",
    index=False,
    encoding="utf-8"
)

print("CSV exportado correctamente.")

#Guardar un archivo .csv

CSV exportado correctamente.
